# BirdCLEF 2026 Inference (No argparse)

Self-contained ensemble + TTA + optional calibration notebook.

In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import json

import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

from birdclef_example.data import SoundscapeSampler
from birdclef_example.model import SimpleCNN

# =========================
# Config (edit these)
# =========================
COMP_INPUT = Path('/kaggle/input/birdclef-2026')
MODEL_INPUT = Path('/kaggle/input/YOUR_MODEL_DATASET')
WORK_DIR = Path('/kaggle/working')

SOUNDSCAPE_DIR = COMP_INPUT / 'test_soundscapes'
METADATA_CSV = COMP_INPUT / 'sample_submission.csv'
OUTPUT_CSV = WORK_DIR / 'submission.csv'

MODEL_PATHS = [
    MODEL_INPUT / 'best_model_fold1.pt',
    MODEL_INPUT / 'best_model_fold2.pt',
    MODEL_INPUT / 'best_model_fold3.pt',
    MODEL_INPUT / 'best_model_fold4.pt',
    MODEL_INPUT / 'best_model_fold5.pt',
]
LABEL_MAP_PATH = MODEL_INPUT / 'label_map.json'
CALIBRATION_JSON = MODEL_INPUT / 'calibration_v2.json'  # optional

SAMPLE_RATE = 32000
SEGMENT_DURATION = 5.0
HOP_DURATION = 5.0
BATCH_SIZE = 32
PRELOAD_WORKERS = 8
USE_BF16 = True
TTA_OFFSETS = [0.0, 1.25, 2.5]

# Optional fallback model config (used only if checkpoint has no model_config)
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

# =========================
# Helpers
# =========================
def load_calibration(calibration_path: Optional[Path]) -> Optional[Dict[str, Dict[str, float]]]:
    if calibration_path is None or not calibration_path.exists():
        return None
    raw = json.loads(calibration_path.read_text())
    if 'labels' not in raw or 'a' not in raw or 'b' not in raw:
        raise ValueError('Calibration JSON must contain keys: labels, a, b')
    labels = raw['labels']
    a_values = raw['a']
    b_values = raw['b']
    if len(labels) != len(a_values) or len(labels) != len(b_values):
        raise ValueError('Calibration arrays labels/a/b must have the same length')
    return {str(lbl): {'a': float(a), 'b': float(b)} for lbl, a, b in zip(labels, a_values, b_values)}

def iterate_batches(segments: Iterable[Tuple[int, torch.Tensor]], batch_size: int):
    end_buf, wav_buf = [], []
    for end_seconds, waveform in segments:
        end_buf.append(end_seconds)
        wav_buf.append(waveform)
        if len(wav_buf) == batch_size:
            yield end_buf, torch.stack(wav_buf)
            end_buf, wav_buf = [], []
    if wav_buf:
        yield end_buf, torch.stack(wav_buf)

def iterate_aligned_segments(waveform: torch.Tensor, sample_rate: int, segment_duration: float, hop_duration: float, offset_seconds: float):
    segment_samples = int(round(segment_duration * sample_rate))
    hop_samples = int(round(hop_duration * sample_rate))
    offset_samples = int(round(offset_seconds * sample_rate))
    total = waveform.size(1)
    if total == 0:
        return

    start = 0
    max_start_for_full = max(0, total - segment_samples)
    while start < total:
        shifted_start = start + offset_samples
        if shifted_start < 0:
            shifted_start = 0
        if shifted_start > max_start_for_full:
            shifted_start = max_start_for_full

        end = min(total, shifted_start + segment_samples)
        chunk = waveform[:, shifted_start:end]
        if chunk.size(1) < segment_samples:
            chunk = F.pad(chunk, (0, segment_samples - chunk.size(1)))

        segment_end_seconds = int(round((start + segment_samples) / sample_rate))
        yield segment_end_seconds, chunk
        start += hop_samples

def apply_calibration(probs: torch.Tensor, reverse_labels: List[str], calibration: Optional[Dict[str, Dict[str, float]]]) -> torch.Tensor:
    if calibration is None:
        return probs
    out = probs.clone()
    eps = 1e-6
    for i, label in enumerate(reverse_labels):
        params = calibration.get(label)
        if params is None:
            continue
        p = out[i].clamp(min=eps, max=1.0 - eps)
        logit = torch.log(p / (1.0 - p))
        logit = params['a'] * logit + params['b']
        out[i] = torch.sigmoid(logit)
    return out

def row_id_from_filename(filename: str, end_seconds: int) -> str:
    return f'{Path(filename).stem}_{end_seconds}'

def build_model_from_checkpoint(checkpoint: Dict, label_count: int, device: torch.device) -> torch.nn.Module:
    model_config = checkpoint.get('model_config', {
        'n_classes': label_count,
        'dropout': 0.3,
        'sample_rate': SAMPLE_RATE,
        'n_mels': N_MELS,
        'n_fft': N_FFT,
        'hop_length': HOP_LENGTH,
        'embed_dim': 256,
        'num_heads': 8,
        'num_layers': 4,
        'token_grid_size': 22,
        'pooling': 'hybrid',
        'freq_mask_param': 12,
        'time_mask_param': 24,
        'specaugment_masks': 2,
    })
    model = SimpleCNN(**model_config)
    model.load_state_dict(checkpoint['model_state'])
    model.to(device)
    model.eval()
    return model

def predict_soundscape(models: List[torch.nn.Module], sampler: SoundscapeSampler, soundscape_path: Path, device: torch.device) -> List[Tuple[int, torch.Tensor]]:
    waveform = sampler._load_cached_audio(soundscape_path)
    end_to_sum: Dict[int, torch.Tensor] = {}
    end_to_count: Dict[int, int] = {}
    amp_enabled = USE_BF16 and device.type == 'cuda'

    with torch.no_grad():
        for model in models:
            for offset in TTA_OFFSETS:
                segments = iterate_aligned_segments(
                    waveform=waveform,
                    sample_rate=sampler.sample_rate,
                    segment_duration=sampler.duration,
                    hop_duration=sampler.hop_samples / sampler.sample_rate,
                    offset_seconds=offset,
                )
                for end_seconds, batch in iterate_batches(segments, BATCH_SIZE):
                    batch = batch.to(device, non_blocking=True)
                    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp_enabled):
                        probs = torch.sigmoid(model(batch)).float().cpu()
                    for i, end_second in enumerate(end_seconds):
                        if end_second not in end_to_sum:
                            end_to_sum[end_second] = probs[i].clone()
                            end_to_count[end_second] = 1
                        else:
                            end_to_sum[end_second] += probs[i]
                            end_to_count[end_second] += 1

    outputs = []
    for end_second in sorted(end_to_sum.keys()):
        outputs.append((end_second, end_to_sum[end_second] / float(end_to_count[end_second])))
    return outputs


In [ ]:
print('Checking inputs...')
print('SOUNDSCAPE_DIR:', SOUNDSCAPE_DIR, SOUNDSCAPE_DIR.exists())
print('METADATA_CSV:', METADATA_CSV, METADATA_CSV.exists())
print('LABEL_MAP_PATH:', LABEL_MAP_PATH, LABEL_MAP_PATH.exists())
print('CALIBRATION_JSON exists:', CALIBRATION_JSON.exists())
for p in MODEL_PATHS:
    print(p, p.exists())

missing = [p for p in MODEL_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing model checkpoints: {missing}')
if not SOUNDSCAPE_DIR.exists() or not METADATA_CSV.exists() or not LABEL_MAP_PATH.exists():
    raise FileNotFoundError('One or more required paths are missing')

checkpoints = [torch.load(path, map_location='cpu') for path in MODEL_PATHS]
if LABEL_MAP_PATH.exists():
    label_map = json.loads(LABEL_MAP_PATH.read_text())
else:
    label_map = checkpoints[0].get('label_map', {})
if not label_map:
    raise ValueError('No label map available')

reverse_labels = sorted(label_map.keys(), key=lambda label: label_map[label])
calibration = load_calibration(CALIBRATION_JSON)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
models = [build_model_from_checkpoint(ckpt, len(label_map), device) for ckpt in checkpoints]

sampler = SoundscapeSampler(
    sample_rate=SAMPLE_RATE,
    duration=SEGMENT_DURATION,
    hop=HOP_DURATION,
    preload_audio=True,
    preload_workers=PRELOAD_WORKERS,
)

metadata = pd.read_csv(METADATA_CSV)
if 'filename' in metadata.columns:
    filenames = metadata['filename'].dropna().astype(str).drop_duplicates().tolist()
elif 'row_id' in metadata.columns:
    filenames = (
        metadata['row_id']
        .dropna()
        .astype(str)
        .str.replace(r'_[0-9]+$', '', regex=True)
        .drop_duplicates()
        .map(lambda stem: f'{stem}.ogg')
        .tolist()
    )
else:
    raise ValueError('metadata must contain filename or row_id column')

print(f'Using {len(models)} models')
print('TTA_OFFSETS:', TTA_OFFSETS)

predictions = []
for filename in tqdm(filenames, desc='Predict'):
    filepath = SOUNDSCAPE_DIR / filename
    if not filepath.exists():
        raise FileNotFoundError(f'Missing soundscape {filepath}')

    for end_seconds, probs in predict_soundscape(models, sampler, filepath, device):
        probs = apply_calibration(probs, reverse_labels, calibration)
        row_dict = {'row_id': row_id_from_filename(filename, end_seconds)}
        row_dict.update({label: float(probs[idx].item()) for idx, label in enumerate(reverse_labels)})
        predictions.append(row_dict)

if not predictions:
    raise RuntimeError('No predictions generated')

submission = pd.DataFrame(predictions)
if 'row_id' in metadata.columns:
    submission = metadata[['row_id']].merge(submission, on='row_id', how='left')
submission = submission[['row_id'] + reverse_labels]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)

# Sanity checks
sample = pd.read_csv(METADATA_CSV)
sub = pd.read_csv(OUTPUT_CSV)
assert sub.shape[0] == sample.shape[0], 'Row count mismatch'
assert list(sub.columns) == list(sample.columns), 'Column order mismatch'
assert not sub.isna().any().any(), 'NaNs in submission'
probs = sub.drop(columns=['row_id'])
assert ((probs >= 0.0) & (probs <= 1.0)).all().all(), 'Probabilities out of [0,1]'
print('Submission checks passed')
sub.head()
